# Monitoring Seasonal Snow-Cover Variability in the Western Everest Region Using Sentinel-2, Unsupervised Classification and Gaussian Process Gap-Filling

## Notebook 1: Sentinel-2 Data Acquisition

This notebook downloads the Sentinel-2 Level-2A imagery used as the input dataset for this project. The study focuses on a selected area in the western Everest region, and the monthly satellite scenes are used to examine monthly changes in snow cover during 2025.

The workflow below defines the study area, searches the Copernicus Data Space for suitable Sentinel-2 data, selects one low-cloud scene for each month, downloads the selected products, and saves their metadata.


## 1. Workspace setup

This step prepares the working environment for the data-download workflow. Google Drive is mounted so that the downloaded Sentinel-2 products and metadata files can be saved permanently outside the temporary Colab session.

The required Python packages are also imported, and the main project folders are created. These folders can help organise the raw Sentinel-2 downloads, processed files, metadata tables, and figures produced during the project.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import requests
import pandas as pd
import getpass
from IPython.display import display

In [ ]:
project_dir = "/content/drive/MyDrive/GEOL0069/Final_Project"

data_dir = os.path.join(project_dir, "Data")
raw_dir = os.path.join(data_dir, "Raw")
processed_dir = os.path.join(data_dir, "Processed")
metadata_dir = os.path.join(data_dir, "Metadata")
figures_dir = os.path.join(project_dir, "Figures")

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)
os.makedirs(metadata_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

print("Project directory:", project_dir)
print("Raw data directory:", raw_dir)
print("Processed data directory:", processed_dir)
print("Metadata directory:", metadata_dir)
print("Figures directory:", figures_dir)

Project directory: /content/drive/MyDrive/GEOL0069/Final_Project
Raw data directory: /content/drive/MyDrive/GEOL0069/Final_Project/Data/Raw
Processed data directory: /content/drive/MyDrive/GEOL0069/Final_Project/Data/Processed
Metadata directory: /content/drive/MyDrive/GEOL0069/Final_Project/Data/Metadata
Figures directory: /content/drive/MyDrive/GEOL0069/Final_Project/Figures


## 2. Copernicus Data Space authentication

This step prepares the authentication functions required to access the Copernicus Data Space API. The access token is used for catalogue queries and product downloads, while the refresh token allows the session to continue if the access token expires.

Login details should be entered manually or loaded securely, so the notebook would not share the personal username and password of the author, and will leave spaces for individual information filling.

In [ ]:
def get_access_and_refresh_token(username, password):
    url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

    data = {
        "grant_type": "password",
        "username": username,
        "password": password,
        "client_id": "cdse-public",
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    tokens = response.json()
    return tokens["access_token"], tokens["refresh_token"]


def refresh_access_token(refresh_token):
    url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

    data = {
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
        "client_id": "cdse-public",
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    tokens = response.json()
    return tokens["access_token"]


def make_api_request(url, method="GET", data=None, headers=None):
    global access_token
    global refresh_token

    if headers is None:
        headers = {"Authorization": f"Bearer {access_token}"}

    response = requests.request(method, url, json=data, headers=headers)

    if response.status_code in [401, 403]:
        access_token = refresh_access_token(refresh_token)
        headers["Authorization"] = f"Bearer {access_token}"
        response = requests.request(method, url, json=data, headers=headers)

    return response

In [ ]:
username = " " # Please fill in your own username for Copernicus
password = " " # Please fill in your own password for Copernicus
access_token, refresh_token = get_access_and_refresh_token(username, password)

## 3. Query Sentinel-2 Level-2A metadata

This section searches the Copernicus Data Space catalogue for Sentinel-2 Level-2A products covering the selected Everest Area of Interest (AOI) during 2025.

The returned catalogue information is converted into a metadata table saved as a CSV file, including product IDs, acquisition dates, cloud-cover values, processing information, online status, and data-access paths. A monthly count is also calculated to check whether suitable candidate images are available throughout the year.

In [ ]:
# Everest-region AOI is selected because it only includes one tile
# If a different AOI is used, the target Sentinel-2 tile should be updated accordingly.
everest_aoi_wkt = (
    "POLYGON((86.85 28.05, "
    "86.97 28.05, "
    "86.97 28.17, "
    "86.85 28.17, "
    "86.85 28.05))"
)

target_tile = "45RVM"
query_start_date = "2025-01-01"
query_end_date = "2025-12-31"
max_cloud_cover = 80

In [ ]:
def get_attribute(product, attribute_name):
    attributes = product.get("Attributes", [])

    for attribute in attributes:
        if attribute.get("Name") == attribute_name:
            if "Value" in attribute:
                return attribute.get("Value")

            for key, value in attribute.items():
                if key.endswith("Value"):
                    return value

    return None

In [ ]:
from urllib.parse import quote

def query_sentinel2_everest_data(
    start_date,
    end_date,
    token,
    aoi_wkt,
    max_cloud_cover=80,
    target_tile="45RVM"
):
    all_products = []

    filter_string = (
        "Collection/Name eq 'SENTINEL-2' and "
        "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' "
        "and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A') and "
        f"ContentDate/Start ge {start_date}T00:00:00.000Z and "
        f"ContentDate/Start le {end_date}T23:59:59.999Z and "
        f"OData.CSC.Intersects(area=geography'SRID=4326;{aoi_wkt}') and "
        f"Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' "
        f"and att/OData.CSC.DoubleAttribute/Value le {max_cloud_cover})"
    )

    encoded_filter = quote(filter_string, safe="()/='$:;, ")

    next_url = (
        "https://catalogue.dataspace.copernicus.eu/odata/v1/Products?"
        f"$filter={encoded_filter}&"
        "$expand=Attributes&"
        "$top=1000"
    )

    headers = {"Authorization": f"Bearer {token}"}

    while next_url:
        response = make_api_request(next_url, headers=headers)

        if response.status_code != 200:
            print("Error fetching data:")
            print("Status code:", response.status_code)
            print(response.text[:1000])
            break

        response_json = response.json()
        products = response_json.get("value", [])
        all_products.extend(products)

        next_url = response_json.get("@odata.nextLink")

    metadata_records = []

    for product in all_products:
        name = product.get("Name")

        if name is None:
            continue

        if "MSIL2A" not in name:
            continue

        if f"T{target_tile}" not in name:
            continue

        content_date = product.get("ContentDate", {})

        record = {
            "Id": product.get("Id"),
            "Name": name,
            "ContentDate_Start": content_date.get("Start"),
            "ContentDate_End": content_date.get("End"),
            "cloudCover": get_attribute(product, "cloudCover"),
            "productType": get_attribute(product, "productType"),
            "processingLevel": get_attribute(product, "processingLevel"),
            "tile": target_tile,
            "S3Path": product.get("S3Path"),
            "Online": product.get("Online"),
            "EvictionDate": product.get("EvictionDate")
        }

        metadata_records.append(record)

    metadata_df = pd.DataFrame(metadata_records)

    if len(metadata_df) > 0:
        metadata_df["ContentDate_Start"] = pd.to_datetime(metadata_df["ContentDate_Start"])
        metadata_df["ContentDate_End"] = pd.to_datetime(metadata_df["ContentDate_End"])
        metadata_df["cloudCover"] = pd.to_numeric(metadata_df["cloudCover"], errors="coerce")
        metadata_df = metadata_df.sort_values("ContentDate_Start").reset_index(drop=True)

    return metadata_df

In [ ]:
sentinel2_metadata_2025 = query_sentinel2_everest_data(
    start_date=query_start_date,
    end_date=query_end_date,
    token=access_token,
    aoi_wkt=everest_aoi_wkt,
    max_cloud_cover=max_cloud_cover,
    target_tile=target_tile
)

print("Number of Sentinel-2 L2A products found:", len(sentinel2_metadata_2025))

display(sentinel2_metadata_2025.head())

Number of Sentinel-2 L2A products found: 79


,Id,Name,ContentDate_Start,ContentDate_End,cloudCover,productType,processingLevel,tile,S3Path,Online,EvictionDate
0,b384fb7e-3ce0-4c8b-ac0c-26513b49173d,S2B_MSIL2A_20250101T045119_N0511_R076_T45RVM_2...,2025-01-01 04:51:19.024000+00:00,2025-01-01 04:51:19.024000+00:00,0.072989,S2MSI2A,S2MSI2A,45RVM,/eodata/Sentinel-2/MSI/L2A/2025/01/01/S2B_MSIL...,True,9999-12-31T23:59:59.999999Z
1,6c9ad95b-8d2e-43fc-91cd-89cb91721b7a,S2A_MSIL2A_20250106T045201_N0511_R076_T45RVM_2...,2025-01-06 04:52:01.024000+00:00,2025-01-06 04:52:01.024000+00:00,22.969984,S2MSI2A,S2MSI2A,45RVM,/eodata/Sentinel-2/MSI/L2A/2025/01/06/S2A_MSIL...,True,9999-12-31T23:59:59.999999Z
2,a3444e7f-e9e3-47cf-aae6-e447c3011c56,S2B_MSIL2A_20250111T045059_N0511_R076_T45RVM_2...,2025-01-11 04:50:59.024000+00:00,2025-01-11 04:50:59.024000+00:00,0.635562,S2MSI2A,S2MSI2A,45RVM,/eodata/Sentinel-2/MSI/L2A/2025/01/11/S2B_MSIL...,True,9999-12-31T23:59:59.999999Z
3,16f3fee9-8b92-4d94-8e54-0be17073859b,S2A_MSIL2A_20250116T045131_N0511_R076_T45RVM_2...,2025-01-16 04:51:31.024000+00:00,2025-01-16 04:51:31.024000+00:00,51.383799,S2MSI2A,S2MSI2A,45RVM,/eodata/Sentinel-2/MSI/L2A/2025/01/16/S2A_MSIL...,True,9999-12-31T23:59:59.999999Z
4,4e578e1c-bc0c-4d45-a7e0-d2546187f1e5,S2B_MSIL2A_20250121T045019_N0511_R076_T45RVM_2...,2025-01-21 04:50:19.024000+00:00,2025-01-21 04:50:19.024000+00:00,0.057724,S2MSI2A,S2MSI2A,45RVM,/eodata/Sentinel-2/MSI/L2A/2025/01/21/S2B_MSIL...,True,9999-12-31T23:59:59.999999Z


In [ ]:
sentinel2_metadata_2025["month"] = sentinel2_metadata_2025["ContentDate_Start"].dt.month

monthly_counts = (
    sentinel2_metadata_2025
    .groupby("month")
    .size()
    .reset_index(name="product_count")
)

display(monthly_counts)

,month,product_count
0,1,7
1,2,5
2,3,7
3,4,7
4,5,8
5,6,5
6,7,7
7,8,5
8,9,7
9,10,6


In [ ]:
metadata_output_path = os.path.join(
    metadata_dir,
    "sentinel2_metadata_2025_45RVM.csv"
)

sentinel2_metadata_2025.to_csv(metadata_output_path, index=False)

print("Metadata table saved to:")
print(metadata_output_path)

Metadata table saved to:
/content/drive/MyDrive/GEOL0069/Final_Project/Data/Metadata/sentinel2_metadata_2025_45RVM.csv


## 4. Select one Sentinel-2 product per month

The catalogue query returns multiple Sentinel-2 products for many months. To build a consistent monthly dataset, one representative Level-2A scene is selected for each month of 2025, with the lowest reported cloud-cover value in that month.

This reduces the full metadata table to 12 monthly products, which are saved separately and used as the input list for downloading the `.SAFE` files.

In [ ]:
metadata_path = os.path.join(
    metadata_dir,
    "sentinel2_metadata_2025_45RVM.csv"
)

sentinel2_metadata_2025 = pd.read_csv(metadata_path)

sentinel2_metadata_2025["ContentDate_Start"] = pd.to_datetime(
    sentinel2_metadata_2025["ContentDate_Start"]
)

sentinel2_metadata_2025["cloudCover"] = pd.to_numeric(
    sentinel2_metadata_2025["cloudCover"],
    errors="coerce"
)

display(sentinel2_metadata_2025.head())

,Id,Name,ContentDate_Start,ContentDate_End,cloudCover,productType,processingLevel,tile,S3Path,Online,EvictionDate,month
0,b384fb7e-3ce0-4c8b-ac0c-26513b49173d,S2B_MSIL2A_20250101T045119_N0511_R076_T45RVM_2...,2025-01-01 04:51:19.024000+00:00,2025-01-01 04:51:19.024000+00:00,0.072989,S2MSI2A,S2MSI2A,45RVM,/eodata/Sentinel-2/MSI/L2A/2025/01/01/S2B_MSIL...,True,9999-12-31T23:59:59.999999Z,1
1,6c9ad95b-8d2e-43fc-91cd-89cb91721b7a,S2A_MSIL2A_20250106T045201_N0511_R076_T45RVM_2...,2025-01-06 04:52:01.024000+00:00,2025-01-06 04:52:01.024000+00:00,22.969984,S2MSI2A,S2MSI2A,45RVM,/eodata/Sentinel-2/MSI/L2A/2025/01/06/S2A_MSIL...,True,9999-12-31T23:59:59.999999Z,1
2,a3444e7f-e9e3-47cf-aae6-e447c3011c56,S2B_MSIL2A_20250111T045059_N0511_R076_T45RVM_2...,2025-01-11 04:50:59.024000+00:00,2025-01-11 04:50:59.024000+00:00,0.635562,S2MSI2A,S2MSI2A,45RVM,/eodata/Sentinel-2/MSI/L2A/2025/01/11/S2B_MSIL...,True,9999-12-31T23:59:59.999999Z,1
3,16f3fee9-8b92-4d94-8e54-0be17073859b,S2A_MSIL2A_20250116T045131_N0511_R076_T45RVM_2...,2025-01-16 04:51:31.024000+00:00,2025-01-16 04:51:31.024000+00:00,51.383799,S2MSI2A,S2MSI2A,45RVM,/eodata/Sentinel-2/MSI/L2A/2025/01/16/S2A_MSIL...,True,9999-12-31T23:59:59.999999Z,1
4,4e578e1c-bc0c-4d45-a7e0-d2546187f1e5,S2B_MSIL2A_20250121T045019_N0511_R076_T45RVM_2...,2025-01-21 04:50:19.024000+00:00,2025-01-21 04:50:19.024000+00:00,0.057724,S2MSI2A,S2MSI2A,45RVM,/eodata/Sentinel-2/MSI/L2A/2025/01/21/S2B_MSIL...,True,9999-12-31T23:59:59.999999Z,1


In [ ]:
sentinel2_metadata_2025["date"] = sentinel2_metadata_2025["ContentDate_Start"].dt.date
sentinel2_metadata_2025["month"] = sentinel2_metadata_2025["ContentDate_Start"].dt.month

display(
    sentinel2_metadata_2025[
        ["month", "date", "Name", "cloudCover", "tile", "Online"]
    ].head()
)

,month,date,Name,cloudCover,tile,Online
0,1,2025-01-01,S2B_MSIL2A_20250101T045119_N0511_R076_T45RVM_2...,0.072989,45RVM,True
1,1,2025-01-06,S2A_MSIL2A_20250106T045201_N0511_R076_T45RVM_2...,22.969984,45RVM,True
2,1,2025-01-11,S2B_MSIL2A_20250111T045059_N0511_R076_T45RVM_2...,0.635562,45RVM,True
3,1,2025-01-16,S2A_MSIL2A_20250116T045131_N0511_R076_T45RVM_2...,51.383799,45RVM,True
4,1,2025-01-21,S2B_MSIL2A_20250121T045019_N0511_R076_T45RVM_2...,0.057724,45RVM,True


In [ ]:
selected_monthly_products = (
    sentinel2_metadata_2025
    .dropna(subset=["cloudCover"])
    .sort_values(["month", "cloudCover", "ContentDate_Start"])
    .groupby("month", as_index=False)
    .first()
)

selected_monthly_products = selected_monthly_products.rename(
    columns={
        "Id": "product_id",
        "Name": "product_name",
        "cloudCover": "cloud_cover",
        "tile": "mgrs_tile"
    }
)

selected_monthly_products["download_status"] = "not_downloaded"

selected_monthly_products = selected_monthly_products[
    [
        "month",
        "date",
        "product_id",
        "product_name",
        "cloud_cover",
        "mgrs_tile",
        "Online",
        "download_status",
        "S3Path"
    ]
]

display(selected_monthly_products)

,month,date,product_id,product_name,cloud_cover,mgrs_tile,Online,download_status,S3Path
0,1,2025-01-31,a940528c-91d5-485c-a17c-465935284530,S2B_MSIL2A_20250131T044939_N0511_R076_T45RVM_2...,0.002054,45RVM,True,not_downloaded,/eodata/Sentinel-2/MSI/L2A/2025/01/31/S2B_MSIL...
1,2,2025-02-10,0b8d069a-6bc9-4998-ab00-d94de4d69700,S2B_MSIL2A_20250210T044839_N0511_R076_T45RVM_2...,2.521034,45RVM,True,not_downloaded,/eodata/Sentinel-2/MSI/L2A/2025/02/10/S2B_MSIL...
2,3,2025-03-27,8b97a93f-be3a-4c89-bb7a-2d3530ccd3bd,S2C_MSIL2A_20250327T044721_N0511_R076_T45RVM_2...,0.010100,45RVM,True,not_downloaded,/eodata/Sentinel-2/MSI/L2A/2025/03/27/S2C_MSIL...
3,4,2025-04-21,9c63c11b-8120-4e2e-82d1-be8f01d1cd43,S2B_MSIL2A_20250421T044659_N0511_R076_T45RVM_2...,0.050823,45RVM,True,not_downloaded,/eodata/Sentinel-2/MSI/L2A/2025/04/21/S2B_MSIL...
4,5,2025-05-08,374dc337-8882-41e5-91a4-e0ca96d79418,S2A_MSIL2A_20250508T045231_N0511_R076_T45RVM_2...,19.816066,45RVM,True,not_downloaded,/eodata/Sentinel-2/MSI/L2A/2025/05/08/S2A_MSIL...
5,6,2025-06-10,5935ac4b-7fe6-40b8-8d50-7509baa71d77,S2B_MSIL2A_20250610T044659_N0511_R076_T45RVM_2...,1.850641,45RVM,True,not_downloaded,/eodata/Sentinel-2/MSI/L2A/2025/06/10/S2B_MSIL...
6,7,2025-07-07,5424cd5d-7568-4fd6-a277-fa46ed7a2556,S2A_MSIL2A_20250707T045241_N0511_R076_T45RVM_2...,12.159456,45RVM,True,not_downloaded,/eodata/Sentinel-2/MSI/L2A/2025/07/07/S2A_MSIL...
7,8,2025-08-19,fb972a2b-f5e5-4c72-b17f-7ddf3a0ded86,S2B_MSIL2A_20250819T044659_N0511_R076_T45RVM_2...,42.063248,45RVM,True,not_downloaded,/eodata/Sentinel-2/MSI/L2A/2025/08/19/S2B_MSIL...
8,9,2025-09-05,fe760223-6cf8-43a6-bd98-0094ff33b589,S2A_MSIL2A_20250905T045231_N0511_R076_T45RVM_2...,23.067698,45RVM,True,not_downloaded,/eodata/Sentinel-2/MSI/L2A/2025/09/05/S2A_MSIL...
9,10,2025-10-25,1e3da49c-5323-4210-b31d-e28ac001b454,S2A_MSIL2A_20251025T045241_N0511_R076_T45RVM_2...,0.058397,45RVM,True,not_downloaded,/eodata/Sentinel-2/MSI/L2A/2025/10/25/S2A_MSIL...


In [ ]:
expected_months = set(range(1, 13))
selected_months = set(selected_monthly_products["month"].tolist())

missing_months = sorted(expected_months - selected_months)

print("Number of selected products:", len(selected_monthly_products))
print("Missing months:", missing_months)

display(
    selected_monthly_products[
        ["month", "date", "cloud_cover", "mgrs_tile", "download_status"]
    ]
)

Number of selected products: 12
Missing months: []


,month,date,cloud_cover,mgrs_tile,download_status
0,1,2025-01-31,0.002054,45RVM,not_downloaded
1,2,2025-02-10,2.521034,45RVM,not_downloaded
2,3,2025-03-27,0.010100,45RVM,not_downloaded
3,4,2025-04-21,0.050823,45RVM,not_downloaded
4,5,2025-05-08,19.816066,45RVM,not_downloaded
5,6,2025-06-10,1.850641,45RVM,not_downloaded
6,7,2025-07-07,12.159456,45RVM,not_downloaded
7,8,2025-08-19,42.063248,45RVM,not_downloaded
8,9,2025-09-05,23.067698,45RVM,not_downloaded
9,10,2025-10-25,0.058397,45RVM,not_downloaded


In [ ]:
selected_products_path = os.path.join(
    metadata_dir,
    "selected_monthly_products_2025.csv"
)

selected_monthly_products.to_csv(selected_products_path, index=False)

print("Selected monthly product table saved to:")
print(selected_products_path)

Selected monthly product table saved to:
/content/drive/MyDrive/GEOL0069/Final_Project/Data/Metadata/selected_monthly_products_2025.csv


## 5. Download and unzip selected Sentinel-2 data

This section downloads the selected Sentinel-2 Level-2A datasets, using the product IDs stored in the monthly metadata table. Each product is downloaded as a `.zip` file and saved in the project `data/raw/` folder on Google Drive.

After downloading, the files are unzipped into `.SAFE` folders, and stored in the `.SAFE` folder inside Data - Raw. These extracted `.SAFE` folders contain the Sentinel-2 band files and scene classification layers which are required for the later snow-cover mapping workflow.

The raw downloaded data files are not included in the GitHub repository because they are 12 large satellite data files. Instead, the notebook and metadata table can record which products were used. If you would like to replicate this project, the dataset can be downloaded onto your computer using the codes in this notebook.

In [ ]:
def download_single_product(product_id, file_name, access_token, download_dir):
    # Copernicus OData product download endpoint
    url = f"https://download.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"

    output_path = os.path.join(download_dir, f"{file_name}.zip")

    # Skip if already downloaded
    if os.path.exists(output_path):
        print(f"Already downloaded: {file_name}.zip")
        return output_path

    print(f"Downloading: {file_name}")

    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    # Use a session so the authorization header is retained correctly
    session = requests.Session()
    session.headers.update(headers)

    response = session.get(url, stream=True)

    if response.status_code == 200:
        with open(output_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)

        print(f"Saved to: {output_path}")
        return output_path

    else:
        print(f"Failed to download: {file_name}")
        print(f"Status code: {response.status_code}")
        print(response.text[:500])
        return None

In [ ]:
download_dir = raw_dir

downloaded_files = []

for index, row in selected_monthly_products.iterrows():
    product_id = row["product_id"]
    file_name = row["product_name"]

    print(f"\nMonth {int(row['month']):02d}: {row['date']}")
    print(f"Cloud cover: {row['cloud_cover']:.3f}%")

    output_path = download_single_product(
        product_id=product_id,
        file_name=file_name,
        access_token=access_token,
        download_dir=download_dir
    )

    downloaded_files.append(output_path)


Month 01: 2025-01-31
Cloud cover: 0.002%
Already downloaded: S2B_MSIL2A_20250131T044939_N0511_R076_T45RVM_20250131T082914.SAFE.zip

Month 02: 2025-02-10
Cloud cover: 2.521%
Already downloaded: S2B_MSIL2A_20250210T044839_N0511_R076_T45RVM_20250210T065139.SAFE.zip

Month 03: 2025-03-27
Cloud cover: 0.010%
Already downloaded: S2C_MSIL2A_20250327T044721_N0511_R076_T45RVM_20250327T094313.SAFE.zip

Month 04: 2025-04-21
Cloud cover: 0.051%
Already downloaded: S2B_MSIL2A_20250421T044659_N0511_R076_T45RVM_20250421T065233.SAFE.zip

Month 05: 2025-05-08
Cloud cover: 19.816%
Already downloaded: S2A_MSIL2A_20250508T045231_N0511_R076_T45RVM_20250508T072516.SAFE.zip

Month 06: 2025-06-10
Cloud cover: 1.851%
Already downloaded: S2B_MSIL2A_20250610T044659_N0511_R076_T45RVM_20250610T070750.SAFE.zip

Month 07: 2025-07-07
Cloud cover: 12.159%
Already downloaded: S2A_MSIL2A_20250707T045241_N0511_R076_T45RVM_20250707T074417.SAFE.zip

Month 08: 2025-08-19
Cloud cover: 42.063%
Already downloaded: S2B_MSIL2A_

In [ ]:
import zipfile

safe_dir = os.path.join(raw_dir, "SAFE")
os.makedirs(safe_dir, exist_ok=True)

zip_files = [
    f for f in os.listdir(raw_dir)
    if f.endswith(".zip")
]

for zip_file in zip_files:
    zip_path = os.path.join(raw_dir, zip_file)
    safe_name = zip_file.replace(".zip", "")
    safe_path = os.path.join(safe_dir, safe_name)

    if os.path.exists(safe_path):
        print(f"Already extracted: {safe_name}")
        continue

    print(f"Extracting: {zip_file}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(safe_dir)

print("Extraction complete.")

Already extracted: S2B_MSIL2A_20250131T044939_N0511_R076_T45RVM_20250131T082914.SAFE
Already extracted: S2B_MSIL2A_20250210T044839_N0511_R076_T45RVM_20250210T065139.SAFE
Already extracted: S2C_MSIL2A_20250327T044721_N0511_R076_T45RVM_20250327T094313.SAFE
Already extracted: S2B_MSIL2A_20250421T044659_N0511_R076_T45RVM_20250421T065233.SAFE
Already extracted: S2A_MSIL2A_20250508T045231_N0511_R076_T45RVM_20250508T072516.SAFE
Already extracted: S2B_MSIL2A_20250610T044659_N0511_R076_T45RVM_20250610T070750.SAFE
Already extracted: S2A_MSIL2A_20250707T045241_N0511_R076_T45RVM_20250707T074417.SAFE
Already extracted: S2B_MSIL2A_20250819T044659_N0511_R076_T45RVM_20250819T070626.SAFE
Already extracted: S2A_MSIL2A_20250905T045231_N0511_R076_T45RVM_20250905T074230.SAFE
Already extracted: S2A_MSIL2A_20251025T045241_N0511_R076_T45RVM_20251025T075715.SAFE
Already extracted: S2B_MSIL2A_20251117T044959_N0511_R076_T45RVM_20251117T065411.SAFE
Already extracted: S2B_MSIL2A_20251217T045119_N0511_R076_T45RVM_2

In [ ]:
safe_folders = [
    f for f in os.listdir(safe_dir)
    if f.endswith(".SAFE")
]

print("Number of extracted SAFE folders:", len(safe_folders))

for folder in safe_folders:
    print(folder)

Number of extracted SAFE folders: 12
S2B_MSIL2A_20250131T044939_N0511_R076_T45RVM_20250131T082914.SAFE
S2B_MSIL2A_20250210T044839_N0511_R076_T45RVM_20250210T065139.SAFE
S2C_MSIL2A_20250327T044721_N0511_R076_T45RVM_20250327T094313.SAFE
S2B_MSIL2A_20250421T044659_N0511_R076_T45RVM_20250421T065233.SAFE
S2A_MSIL2A_20250508T045231_N0511_R076_T45RVM_20250508T072516.SAFE
S2B_MSIL2A_20250610T044659_N0511_R076_T45RVM_20250610T070750.SAFE
S2A_MSIL2A_20250707T045241_N0511_R076_T45RVM_20250707T074417.SAFE
S2B_MSIL2A_20250819T044659_N0511_R076_T45RVM_20250819T070626.SAFE
S2A_MSIL2A_20250905T045231_N0511_R076_T45RVM_20250905T074230.SAFE
S2A_MSIL2A_20251025T045241_N0511_R076_T45RVM_20251025T075715.SAFE
S2B_MSIL2A_20251117T044959_N0511_R076_T45RVM_20251117T065411.SAFE
S2B_MSIL2A_20251217T045119_N0511_R076_T45RVM_20251217T063558.SAFE
